# End-to-End GRPO Pipeline (OpenEnv + Unsloth + TRL)

This notebook trains your existing SFT model in your already-running OpenEnv HF Space.

What it does:
1. Connect to remote env Space using the OpenEnv client (`.sync()` usage).
2. Build a GRPO prompt dataset from environment rollouts.
3. Load your model `MHussain17/sft-qwen` with Unsloth.
4. Train with TRL `GRPOTrainer` using an environment-based reward function.
5. Save and optionally push RL checkpoint to HF Hub.

In [ ]:
!pip -q install --upgrade unsloth trl transformers datasets accelerate peft bitsandbytes huggingface_hub "openenv-core[core]>=0.2.2"

In [ ]:
import os

# --- Required ---
MODEL_ID = "MHussain17/sft-qwen"
ENV_SPACE_URL = "https://YOUR-ENV-SPACE.hf.space"  # change this

# HF token (set in Colab/Notebook secrets or env var)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# If MODEL_ID is adapter-only and fails to load directly, set base model here.
BASE_MODEL_ID = "Qwen/Qwen3.5-0.8B"

# --- Training/output ---
# Checkpoints + final save go here. On Colab, mount Drive and use My Drive.
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_SUBDIR = "energy_grid_rl/grpo_openenv"  # folder under My Drive

LOCAL_OUTPUT_DIR = "outputs/grpo_openenv"
OUTPUT_DIR = LOCAL_OUTPUT_DIR

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        OUTPUT_DIR = os.path.join("/content/drive/MyDrive", GOOGLE_DRIVE_SUBDIR)
        print("Checkpoints will save to Google Drive:", OUTPUT_DIR)
    except ImportError:
        print("Not running in Colab (no google.colab); using local folder:", LOCAL_OUTPUT_DIR)
        OUTPUT_DIR = LOCAL_OUTPUT_DIR
else:
    OUTPUT_DIR = LOCAL_OUTPUT_DIR

OUTPUT_MODEL_ID = "MHussain17/sft-qwen-grpo-openenv"  # change if needed
PUSH_TO_HUB = True

# Checkpoint frequency (used by GRPOConfig below)
SAVE_EVERY_STEPS = 30

# --- GRPO data collection ---
PROMPT_EPISODES = 400
MAX_EPISODE_STEPS = 24
SEED = 42

# --- Sequence/VRAM knobs ---
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("Config loaded")

In [ ]:
import random
import re
import numpy as np
from datasets import Dataset
from huggingface_hub import login, list_repo_files
from peft import PeftModel
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

from client import EnergyGridEnv
from models import GridAction

try:
    from inference import _SYSTEM_PROMPT, _heuristic_action
except Exception:
    _SYSTEM_PROMPT = (
        "You are an expert energy grid dispatch operator managing a 500kW microgrid. "
        "Output one action in this exact format: ### Action: [bess, hospital, industrial, residential]."
    )

    def _heuristic_action(obs):
        # Fallback simple heuristic
        return {"bess": 0, "hospital": 0, "industrial": 0, "residential": 0}

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF login: ok")
else:
    print("HF_TOKEN not set (ok if model is public and you do not push).")

def format_prompt(obs):
    vec = [round(v, 3) for v in obs.to_vector()]
    user_prompt = (
        f"Obs vector (22 values): {vec}\n"
        f"solar_swan={obs.solar_swan_active:.0f} wind_swan={obs.wind_swan_active:.0f} "
        f"soc={obs.battery_soc:.2f} freq_norm={obs.frequency_norm:.3f}\n"
        f"hosp_ratio={obs.hosp_served_ratio:.3f} ind_ratio={obs.ind_served_ratio:.3f} "
        f"res_ratio={obs.res_served_ratio:.3f}\n"
        "What is your dispatch decision?"
    )
    return _SYSTEM_PROMPT + "\n\n" + user_prompt

ACTION_RE = re.compile(r"###\s*Action:\s*\[(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\]")

def completion_to_text(c):
    # TRL can pass plain strings or chat-format lists.
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        parts = []
        for msg in c:
            if isinstance(msg, dict) and "content" in msg:
                parts.append(str(msg["content"]))
        return "\n".join(parts)
    return str(c)

def parse_action(text):
    m = ACTION_RE.search(text)
    if not m:
        return None
    b, h, i, r = [int(x) for x in m.groups()]
    if not (0 <= b <= 5 and 0 <= h <= 1 and 0 <= i <= 3 and 0 <= r <= 3):
        return None
    return GridAction(bess=b, hospital=h, industrial=i, residential=r)

In [ ]:
# Quick OpenEnv connectivity check (latest docs pattern: .sync() in notebooks)
with EnergyGridEnv(base_url=ENV_SPACE_URL).sync() as env:
    result = env.reset(seed=SEED, scenario=0)
    obs = result.observation
    act = _heuristic_action(obs)
    result2 = env.step(GridAction(**act))
    print("Reset+step ok | reward:", result2.reward, "done:", result2.done)

In [ ]:
def build_grpo_prompt_dataset(episodes=PROMPT_EPISODES, max_steps=MAX_EPISODE_STEPS, seed=SEED):
    rng = random.Random(seed)
    rows = []
    scenarios = [0, 1, 2, 3, 4]

    with EnergyGridEnv(base_url=ENV_SPACE_URL).sync() as env:
        for ep in range(episodes):
            scenario = scenarios[ep % len(scenarios)]
            episode_seed = seed + ep * 19

            result = env.reset(seed=episode_seed, scenario=scenario)
            obs = result.observation

            for step_idx in range(max_steps):
                rows.append(
                    {
                        "prompt": format_prompt(obs),
                        "seed": int(episode_seed),
                        "scenario": int(scenario),
                        "step_idx": int(step_idx),
                    }
                )

                # Advance to next state using heuristic to cover realistic state manifold.
                action_dict = _heuristic_action(obs)
                result = env.step(GridAction(**action_dict))
                obs = result.observation
                if result.done:
                    break

            if (ep + 1) % 50 == 0:
                print(f"Collected episodes: {ep + 1}/{episodes} | prompts so far: {len(rows)}")

    rng.shuffle(rows)
    ds = Dataset.from_list(rows)
    print(ds)
    return ds

train_dataset = build_grpo_prompt_dataset()
train_dataset[0]

In [ ]:
def load_model_for_grpo(model_id=MODEL_ID, base_model_id=BASE_MODEL_ID):
    files = set(list_repo_files(model_id))
    is_adapter_only = "adapter_config.json" in files

    if is_adapter_only:
        print(f"{model_id} appears adapter-only; loading base {base_model_id} then adapter.")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_id,
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=LOAD_IN_4BIT,
        )
        model = PeftModel.from_pretrained(model, model_id)
    else:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=LOAD_IN_4BIT,
        )

    # If no trainable adapters are present, attach LoRA adapters for GRPO updates.
    if not hasattr(model, "peft_config"):
        model = FastLanguageModel.get_peft_model(
            model,
            r=8,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=16,
            lora_dropout=0.0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=SEED,
        )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer

model, tokenizer = load_model_for_grpo()
print("Model loaded")

In [ ]:
def env_reward(
    completions,
    seed,
    scenario,
    step_idx,
    trainer_state=None,
    **kwargs,
):
    rewards = []
    texts = [completion_to_text(c) for c in completions]

    # One client per reward-batch (more efficient than per-sample connect/disconnect).
    with EnergyGridEnv(base_url=ENV_SPACE_URL).sync() as env:
        for comp_text, sd, sc, st in zip(texts, seed, scenario, step_idx):
            action = parse_action(comp_text)
            if action is None:
                rewards.append(-500.0)
                continue

            try:
                result = env.reset(seed=int(sd), scenario=int(sc))
                obs = result.observation

                # Replay trajectory to the prompt state.
                for _ in range(int(st)):
                    heur = _heuristic_action(obs)
                    result = env.step(GridAction(**heur))
                    obs = result.observation
                    if result.done:
                        break

                # Candidate action reward.
                result = env.step(action)
                r = float(result.reward if result.reward is not None else 0.0)
                rewards.append(r)
            except Exception:
                rewards.append(-50.0)

    # Normalize within batch to stabilize GRPO if reward scale shifts by scenario.
    if len(rewards) > 1 and np.std(rewards) > 0:
        arr = np.array(rewards, dtype=np.float32)
        arr = (arr - arr.mean()) / (arr.std() + 1e-8)
        rewards = arr.tolist()

    return rewards

print("Reward function ready")

In [ ]:
# T4-safe GRPO defaults; reduce num_generations/max_completion_length if OOM.
# Checkpoints: every SAVE_EVERY_STEPS (30) to OUTPUT_DIR (Google Drive on Colab if USE_GOOGLE_DRIVE).
grpo_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_generations=4,
    max_prompt_length=320,
    max_completion_length=160,
    max_steps=400,
    save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS,
    save_total_limit=50,
    max_grad_norm=0.1,
    bf16=True,
    report_to="none",
    push_to_hub=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[env_reward],
    args=grpo_args,
    train_dataset=train_dataset,
)

train_result = trainer.train()
train_result

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved local artifacts to", OUTPUT_DIR)

if PUSH_TO_HUB:
    # For Unsloth, pushing merged LoRA format is a practical default.
    model.push_to_hub_merged(
        OUTPUT_MODEL_ID,
        tokenizer,
        save_method="lora",
        token=HF_TOKEN if HF_TOKEN else None,
    )
    print("Pushed to Hub:", OUTPUT_MODEL_ID)

In [ ]:
# Optional: one rollout with the trained policy for sanity check.
from transformers import TextStreamer

def generate_action(obs):
    prompt = format_prompt(obs)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
    )
    txt = tokenizer.decode(out[0], skip_special_tokens=True)
    action = parse_action(txt)
    if action is None:
        action = GridAction(bess=0, hospital=0, industrial=0, residential=0)
    return action, txt

episode_reward = 0.0
with EnergyGridEnv(base_url=ENV_SPACE_URL).sync() as env:
    result = env.reset(seed=123, scenario=2)
    obs = result.observation

    for t in range(24):
        action, raw = generate_action(obs)
        result = env.step(action)
        obs = result.observation
        r = float(result.reward if result.reward is not None else 0.0)
        episode_reward += r
        print(f"step={t+1:02d} action={[action.bess, action.hospital, action.industrial, action.residential]} reward={r:.3f} done={result.done}")
        if result.done:
            break

print("Episode reward:", round(episode_reward, 3))

## Notes

- **Checkpoints:** Training saves to `OUTPUT_DIR` every `SAVE_EVERY_STEPS` (30). On Colab with `USE_GOOGLE_DRIVE = True`, that path is under **Google Drive → My Drive →** `GOOGLE_DRIVE_SUBDIR`.
- If GPU memory is tight, lower `num_generations` to `2` and `max_completion_length` to `96`.
- If training is slow due to remote reward calls, reduce `PROMPT_EPISODES` and `max_steps` initially.
- For stronger RL, you can increase `PROMPT_EPISODES` and `max_steps` once the pipeline is stable.